<a href="https://colab.research.google.com/github/sahitchallakonda18/Sahit/blob/main/Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# 🌍 GEOID / GRAVITY FIELD ANALYZER
# One-cell full-stack prototype for Google Colab
#
# Mathematical method:
#   A = U Σ Vᵀ
#   A_k = U_k Σ_k V_kᵀ
#
# IMPORTANT:
# This project uses a SYNTHETIC gravity-like benchmark field.
# It is NOT actual satellite gravity/geoid data.
# ============================================================

!pip -q install flask

import numpy as np
import json
import threading
import time

from flask import Flask, request, jsonify, render_template_string
from google.colab import output


# ============================================================
# 1. MATHEMATICAL ENGINE
# ============================================================

def generate_gravity_field(n_lat=20, n_lon=36, noise_level=0.0):
    """
    Generate a smooth synthetic gravity-like spatial field.

    This is a mathematical benchmark, NOT a physical geoid model.
    """

    lat = np.linspace(-90, 90, n_lat)
    lon = np.linspace(-180, 180, n_lon)

    lat_r = np.deg2rad(lat)[:, None]
    lon_r = np.deg2rad(lon)[None, :]

    # Smooth spatial benchmark components
    field = (
        9.80
        + 0.08 * np.cos(lat_r) * np.cos(lon_r)
        + 0.04 * np.sin(lat_r) * np.sin(2 * lon_r)
        + 0.025 * np.cos(2 * lat_r) * np.cos(3 * lon_r)
        + 0.015 * np.sin(3 * lat_r)
    )

    if noise_level > 0:
        rng = np.random.default_rng(42)
        field += rng.normal(0, noise_level, field.shape)

    return lat, lon, field


def create_sparse_data(data, missing_rate, seed=42):
    """
    Randomly remove a percentage of measurements.

    True values are retained separately so that reconstruction
    accuracy can be evaluated.
    """

    rng = np.random.default_rng(seed)

    mask = rng.random(data.shape) > missing_rate

    sparse = data.copy()
    sparse[~mask] = np.nan

    return sparse, mask


def low_rank_svd(matrix, rank):
    """
    Perform truncated Singular Value Decomposition.

        A = U Σ Vᵀ

    and reconstruct:

        A_k = U_k Σ_k V_kᵀ
    """

    U, S, VT = np.linalg.svd(matrix, full_matrices=False)

    rank = min(rank, len(S))

    U_k = U[:, :rank]
    S_k = S[:rank]
    VT_k = VT[:rank, :]

    reconstructed = U_k @ np.diag(S_k) @ VT_k

    return reconstructed, S


def iterative_svd_completion(
    sparse_data,
    rank=4,
    iterations=25,
    tolerance=1e-6
):
    """
    Iterative SVD-based low-rank reconstruction.

    Steps:
    1. Initialize missing values using observed mean.
    2. Perform SVD.
    3. Construct rank-k approximation.
    4. Keep observed measurements fixed.
    5. Replace only missing values.
    6. Repeat.
    """

    observed_mask = ~np.isnan(sparse_data)

    # Initial fill
    observed_mean = np.nanmean(sparse_data)

    completed = np.where(
        observed_mask,
        sparse_data,
        observed_mean
    )

    for iteration in range(iterations):

        previous = completed.copy()

        reconstructed, singular_values = low_rank_svd(
            completed,
            rank
        )

        # Keep original observations fixed
        completed = np.where(
            observed_mask,
            sparse_data,
            reconstructed
        )

        change = np.linalg.norm(
            completed - previous
        )

        if change < tolerance:
            break

    return completed, singular_values, iteration + 1


def calculate_metrics(true_data, reconstructed, sparse_data):

    error = reconstructed - true_data

    rmse = np.sqrt(np.mean(error ** 2))

    relative_error = (
        np.linalg.norm(error)
        /
        np.linalg.norm(true_data)
    )

    missing_mask = np.isnan(sparse_data)

    if np.any(missing_mask):
        missing_rmse = np.sqrt(
            np.mean(
                (reconstructed[missing_mask]
                 - true_data[missing_mask]) ** 2
            )
        )
    else:
        missing_rmse = 0.0

    return {
        "rmse": float(rmse),
        "relative_error": float(relative_error),
        "missing_rmse": float(missing_rmse)
    }


def calculate_energy(singular_values, rank):

    total_energy = np.sum(singular_values ** 2)

    captured_energy = np.sum(
        singular_values[:rank] ** 2
    )

    percentage = (
        captured_energy /
        total_energy *
        100
    )

    return float(percentage)


# ============================================================
# 2. FLASK APPLICATION
# ============================================================

app = Flask(__name__)


# ============================================================
# 3. FRONTEND
# ============================================================

HTML = r"""
<!DOCTYPE html>
<html>

<head>

<title>Geoid Field Analyzer</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: Arial, sans-serif;
    background:
        radial-gradient(circle at top left, #17345f, #07111f 45%, #030712);
    color: white;
}

.navbar {
    height: 70px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 0 35px;
    background: rgba(5,12,25,0.8);
    border-bottom: 1px solid rgba(255,255,255,0.1);
}

.logo {
    font-size: 22px;
    font-weight: bold;
}

.logo span {
    color: #55c7ff;
}

.nav {
    color: #aebbd0;
}

.hero {
    text-align: center;
    padding: 45px 20px 25px;
}

.hero h1 {
    font-size: 42px;
    margin: 5px;
}

.hero p {
    color: #aebbd0;
    max-width: 750px;
    margin: 15px auto;
    line-height: 1.6;
}

.container {
    max-width: 1350px;
    margin: auto;
    padding: 20px;
}

.controls {
    background: rgba(255,255,255,0.06);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 18px;
    padding: 25px;
    margin-bottom: 25px;
}

.controls h2 {
    margin-top: 0;
}

.grid {
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(180px, 1fr));
    gap: 15px;
}

label {
    color: #b9c6d9;
    font-size: 14px;
}

input {
    width: 100%;
    margin-top: 7px;
    padding: 12px;
    border-radius: 10px;
    border: 1px solid #34445c;
    background: #101c2e;
    color: white;
}

button {
    margin-top: 20px;
    width: 100%;
    padding: 14px;
    border: none;
    border-radius: 10px;
    background: linear-gradient(90deg, #168cff, #48d4ff);
    color: white;
    font-size: 16px;
    font-weight: bold;
    cursor: pointer;
}

button:hover {
    opacity: 0.9;
}

.metrics {
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(180px, 1fr));
    gap: 15px;
    margin-bottom: 25px;
}

.metric {
    background: rgba(255,255,255,0.07);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 16px;
    padding: 20px;
}

.metric-title {
    color: #9eacc1;
    font-size: 14px;
}

.metric-value {
    font-size: 28px;
    font-weight: bold;
    margin-top: 8px;
}

.chart-grid {
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(450px, 1fr));
    gap: 20px;
}

.chart {
    min-height: 420px;
    background: rgba(255,255,255,0.05);
    border-radius: 16px;
    border: 1px solid rgba(255,255,255,0.08);
}

.info {
    margin-top: 25px;
    padding: 25px;
    border-radius: 16px;
    background: rgba(255,255,255,0.05);
    line-height: 1.7;
    color: #c3cfdf;
}

.formula {
    background: #07101d;
    padding: 15px;
    border-radius: 10px;
    text-align: center;
    font-size: 20px;
    margin: 12px 0;
    color: #69d5ff;
}

.status {
    text-align: center;
    color: #8ea2bd;
    margin: 15px;
}

.footer {
    text-align: center;
    padding: 40px;
    color: #6f8199;
}

</style>

</head>


<body>

<div class="navbar">

    <div class="logo">
        🌍 <span>GEOID</span> FIELD ANALYZER
    </div>

    <div class="nav">
        SVD • Low-Rank Approximation
    </div>

</div>


<div class="hero">

    <h1>Gravity-Like Field Analysis</h1>

    <p>
        Explore how Singular Value Decomposition can represent
        a smooth spatial field using a small number of dominant
        mathematical components.
    </p>

</div>


<div class="container">


<div class="controls">

<h2>⚙️ Analysis Configuration</h2>

<div class="grid">

<div>
<label>Latitude Points</label>
<input id="lat" type="number" value="20" min="5" max="100">
</div>

<div>
<label>Longitude Points</label>
<input id="lon" type="number" value="36" min="5" max="150">
</div>

<div>
<label>Missing Data (%)</label>
<input id="missing" type="number" value="30" min="0" max="90">
</div>

<div>
<label>SVD Rank</label>
<input id="rank" type="number" value="4" min="1" max="20">
</div>

<div>
<label>Iterations</label>
<input id="iterations" type="number" value="25" min="1" max="100">
</div>

</div>

<button onclick="runAnalysis()">
🚀 RUN SVD ANALYSIS
</button>

<div class="status" id="status">
Ready for analysis
</div>

</div>


<div class="metrics">

<div class="metric">
<div class="metric-title">Captured Energy</div>
<div class="metric-value" id="energy">—</div>
</div>

<div class="metric">
<div class="metric-title">RMSE</div>
<div class="metric-value" id="rmse">—</div>
</div>

<div class="metric">
<div class="metric-title">Missing RMSE</div>
<div class="metric-value" id="missingRmse">—</div>
</div>

<div class="metric">
<div class="metric-title">Relative Error</div>
<div class="metric-value" id="relative">—</div>
</div>

<div class="metric">
<div class="metric-title">Iterations</div>
<div class="metric-value" id="iters">—</div>
</div>

</div>


<div class="chart-grid">

<div id="original" class="chart"></div>

<div id="sparse" class="chart"></div>

<div id="reconstructed" class="chart"></div>

<div id="error" class="chart"></div>

<div id="singular" class="chart"></div>

<div id="rankError" class="chart"></div>

</div>


<div class="info">

<h2>📐 Mathematics Behind the Model</h2>

<p>
The spatial field is represented as a matrix:
</p>

<div class="formula">
A ∈ ℝ<sup>m×n</sup>
</div>

<p>
Singular Value Decomposition decomposes the matrix as:
</p>

<div class="formula">
A = UΣVᵀ
</div>

<p>
A rank-k approximation keeps only the largest k singular values:
</p>

<div class="formula">
A<sub>k</sub> = U<sub>k</sub>Σ<sub>k</sub>V<sub>k</sub>ᵀ
</div>

<p>
The experiment also removes a configurable percentage of
measurements and uses an iterative SVD-based low-rank
reconstruction to estimate the missing entries.
</p>

<p>
<strong>Important:</strong>
The benchmark is synthetic and represents a smooth
gravity-like spatial field. It is not actual satellite
gravity data and the output is not a physical geoid model.
</p>

</div>


</div>


<div class="footer">
B.Tech Mathematics Mini Project • SVD • Low-Rank Approximation
</div>


<script>

function makeHeatmap(
    id,
    title,
    matrix,
    lat,
    lon
) {

    let data = [{
        z: matrix,
        x: lon,
        y: lat,
        type: "heatmap",
        colorscale: "Viridis",
        colorbar: {
            title: "Value"
        }
    }];

    let layout = {
        title: title,
        paper_bgcolor: "rgba(0,0,0,0)",
        plot_bgcolor: "rgba(0,0,0,0)",
        font: {
            color: "white"
        },
        xaxis: {
            title: "Longitude (°)"
        },
        yaxis: {
            title: "Latitude (°)"
        }
    };

    Plotly.newPlot(id, data, layout, {
        responsive: true
    });
}


async function runAnalysis() {

    document.getElementById("status").innerText =
        "⏳ Running SVD analysis...";

    const payload = {

        lat_points:
            parseInt(document.getElementById("lat").value),

        lon_points:
            parseInt(document.getElementById("lon").value),

        missing_rate:
            parseFloat(document.getElementById("missing").value) / 100,

        rank:
            parseInt(document.getElementById("rank").value),

        iterations:
            parseInt(document.getElementById("iterations").value)

    };


    try {

        const response = await fetch(
            "/api/analyze",
            {
                method: "POST",
                headers: {
                    "Content-Type": "application/json"
                },
                body: JSON.stringify(payload)
            }
        );


        const result = await response.json();


        if (!response.ok) {
            throw new Error(result.error);
        }


        document.getElementById("energy").innerText =
            result.energy.toFixed(2) + "%";

        document.getElementById("rmse").innerText =
            result.metrics.rmse.toFixed(5);

        document.getElementById("missingRmse").innerText =
            result.metrics.missing_rmse.toFixed(5);

        document.getElementById("relative").innerText =
            result.metrics.relative_error.toFixed(5);

        document.getElementById("iters").innerText =
            result.iterations;


        makeHeatmap(
            "original",
            "🌍 Original Synthetic Field",
            result.original,
            result.lat,
            result.lon
        );


        makeHeatmap(
            "sparse",
            "🕳️ Sparse Measurements",
            result.sparse,
            result.lat,
            result.lon
        );


        makeHeatmap(
            "reconstructed",
            "🔬 SVD Reconstruction",
            result.reconstructed,
            result.lat,
            result.lon
        );


        makeHeatmap(
            "error",
            "📊 Reconstruction Error",
            result.error,
            result.lat,
            result.lon
        );


        Plotly.newPlot(
            "singular",
            [{
                x: result.singular_values.map(
                    (_, i) => i + 1
                ),
                y: result.singular_values,
                type: "bar"
            }],
            {
                title: "Singular Values",
                paper_bgcolor: "rgba(0,0,0,0)",
                plot_bgcolor: "rgba(0,0,0,0)",
                font: {color: "white"},
                xaxis: {
                    title: "Component"
                },
                yaxis: {
                    title: "Singular Value"
                }
            },
            {responsive: true}
        );


        Plotly.newPlot(
            "rankError",
            [{
                x: result.rank_analysis.map(
                    x => x.rank
                ),
                y: result.rank_analysis.map(
                    x => x.rmse
                ),
                mode: "lines+markers",
                type: "scatter"
            }],
            {
                title: "Reconstruction Error vs Rank",
                paper_bgcolor: "rgba(0,0,0,0)",
                plot_bgcolor: "rgba(0,0,0,0)",
                font: {color: "white"},
                xaxis: {
                    title: "Rank k"
                },
                yaxis: {
                    title: "RMSE"
                }
            },
            {responsive: true}
        );


        document.getElementById("status").innerText =
            "✅ Analysis completed successfully";


    } catch(error) {

        document.getElementById("status").innerText =
            "❌ Error: " + error.message;

    }

}

</script>

</body>

</html>
"""


# ============================================================
# 4. FRONTEND ROUTE
# ============================================================

@app.route("/")
def home():
    return render_template_string(HTML)


# ============================================================
# 5. ANALYSIS API
# ============================================================

@app.route("/api/analyze", methods=["POST"])
def analyze():

    try:

        config = request.get_json()

        n_lat = int(config.get("lat_points", 20))
        n_lon = int(config.get("lon_points", 36))

        missing_rate = float(
            config.get("missing_rate", 0.30)
        )

        rank = int(
            config.get("rank", 4)
        )

        iterations = int(
            config.get("iterations", 25)
        )


        # Safety limits
        n_lat = max(5, min(n_lat, 100))
        n_lon = max(5, min(n_lon, 150))

        missing_rate = max(
            0.0,
            min(missing_rate, 0.90)
        )

        rank = max(
            1,
            min(rank, min(n_lat, n_lon))
        )

        iterations = max(
            1,
            min(iterations, 100)
        )


        # ----------------------------------------------------
        # Generate benchmark
        # ----------------------------------------------------

        lat, lon, true_data = generate_gravity_field(
            n_lat,
            n_lon
        )


        # ----------------------------------------------------
        # Create sparse measurements
        # ----------------------------------------------------

        sparse_data, mask = create_sparse_data(
            true_data,
            missing_rate
        )


        # ----------------------------------------------------
        # Iterative SVD completion
        # ----------------------------------------------------

        completed, singular_values, used_iterations = (
            iterative_svd_completion(
                sparse_data,
                rank,
                iterations
            )
        )


        # ----------------------------------------------------
        # Final low-rank approximation
        # ----------------------------------------------------

        reconstructed, final_singular_values = (
            low_rank_svd(
                completed,
                rank
            )
        )


        # Keep observed values exactly fixed
        reconstructed[mask] = sparse_data[mask]


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        metrics = calculate_metrics(
            true_data,
            reconstructed,
            sparse_data
        )


        energy = calculate_energy(
            final_singular_values,
            rank
        )


        # ----------------------------------------------------
        # Error field
        # ----------------------------------------------------

        error_field = (
            reconstructed - true_data
        )


        # ----------------------------------------------------
        # Rank experiment
        # ----------------------------------------------------

        rank_analysis = []

        max_rank = min(
            10,
            min(n_lat, n_lon)
        )

        for r in range(1, max_rank + 1):

            temp_completed, _, _ = (
                iterative_svd_completion(
                    sparse_data,
                    r,
                    iterations
                )
            )

            temp_reconstructed, _ = (
                low_rank_svd(
                    temp_completed,
                    r
                )
            )

            temp_reconstructed[mask] = sparse_data[mask]

            temp_rmse = np.sqrt(
                np.mean(
                    (
                        temp_reconstructed
                        - true_data
                    ) ** 2
                )
            )

            rank_analysis.append({
                "rank": r,
                "rmse": float(temp_rmse)
            })


        # ----------------------------------------------------
        # Convert NaN safely for JSON
        # ----------------------------------------------------

        sparse_display = np.where(
            np.isnan(sparse_data),
            None,
            sparse_data
        ).tolist()


        return jsonify({

            "lat": lat.tolist(),

            "lon": lon.tolist(),

            "original":
                true_data.tolist(),

            "sparse":
                sparse_display,

            "reconstructed":
                reconstructed.tolist(),

            "error":
                error_field.tolist(),

            "singular_values":
                final_singular_values.tolist(),

            "energy":
                energy,

            "metrics":
                metrics,

            "iterations":
                used_iterations,

            "rank_analysis":
                rank_analysis

        })


    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 400


# ============================================================
# 6. START FLASK SERVER
# ============================================================

def run_server():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

time.sleep(2)

print("====================================================")
print("🌍 GEOID FIELD ANALYZER")
print("====================================================")
print("✅ Flask backend started")
print("✅ SVD mathematical engine loaded")
print("✅ Frontend loaded")
print("✅ API endpoint: /api/analyze")
print("====================================================")
print("Opening website...")
print("====================================================")


output.serve_kernel_port_as_iframe(5000, height=1000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🌍 GEOID FIELD ANALYZER
✅ Flask backend started
✅ SVD mathematical engine loaded
✅ Frontend loaded
✅ API endpoint: /api/analyze
Opening website...


<IPython.core.display.Javascript object>